In [1]:
# ============================================================
# S1: WEARABLE-ONLY BASELINE (no PSG, no KD, no artifact gate)
#
#   Zmax EEG ──> Zmax Encoder ──┐
#                                ├── Concat Fusion ──> Classifier
#   E4       ──> E4 Encoder ────┘
#
#   Loss = CE only (weighted by class frequency)
#
# This is the reference point. Once this number is in, the next
# script (S2) adds knowledge distillation from the frozen PSG
# teacher and we compare against THIS baseline to measure the
# benefit of KD.
# ============================================================

import os
import csv
import math
import random
import warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from collections import Counter
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score
warnings.filterwarnings('ignore')

# ============================================================
# PATHS
# ============================================================
STUDENT_DATA_PATH = r"D:\22\AA\preprocess\preprocessed_student_zmax_e4"
SPLIT_REF_PATH    = r"D:\22\AA\preprocess\preprocessed_FFinal"   # reuse existing train/test split
EVAL_PATH         = r"D:\22\AA\evaluation\teacher-student\s1_wearable_baseline"
os.makedirs(EVAL_PATH, exist_ok=True)

LABEL_NAMES = ["Wake", "N1", "N2", "N3", "REM"]
N_EPOCHS    = 30
BATCH_SIZE  = 64
SEEDS       = [42, 123, 256, 789, 999]

D_MODEL = 96
DROPOUT = 0.3

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")
print(f"S1: Wearable-only baseline (Zmax EEG + E4, plain CE, no KD, no gate)")
print(f"Output : {EVAL_PATH}")


# ============================================================
# SPLIT (reuse existing subject-level split; keep only subjects
# that also have student data)
# ============================================================
_train_path = os.path.join(SPLIT_REF_PATH, "_train_subs.npy")
_test_path  = os.path.join(SPLIT_REF_PATH, "_test_subs.npy")
TRAIN_SUBS = np.load(_train_path, allow_pickle=True).tolist()
TEST_SUBS  = np.load(_test_path,  allow_pickle=True).tolist()
print(f"Split loaded -> Train:{len(TRAIN_SUBS)}  Test:{len(TEST_SUBS)}")


# ============================================================
# DATASET (Zmax + E4 only, single center epoch, plain labels)
# ============================================================
class WearableDataset(Dataset):
    def __init__(self, subject_list, data_path):
        self.data  = []
        self.index = []
        label_counter = Counter()

        n_loaded = 0
        for sub in subject_list:
            fp = os.path.join(data_path, f"{sub}.npz")
            if not os.path.exists(fp):
                continue
            with np.load(fp) as d:
                zmax_arr = d['zmax_eeg']    # (N, 2, 1920)
                e4_arr   = d['e4']          # (N, 3, 1920)
                labels   = d['labels'].copy()

            n = min(zmax_arr.shape[0], e4_arr.shape[0], len(labels))
            sub_idx = len(self.data)
            self.data.append((zmax_arr[:n], e4_arr[:n], labels[:n]))
            for i in range(n):
                self.index.append((sub_idx, i))
                label_counter[int(labels[i])] += 1
            n_loaded += 1

        self.label_counts = np.array(
            [label_counter[i] for i in range(5)], dtype=np.float32
        )
        print(f"  Subjects: {n_loaded}   Samples: {len(self.index):,}")

    def __len__(self):
        return len(self.index)

    def __getitem__(self, idx):
        sub_idx, i = self.index[idx]
        zmax_arr, e4_arr, labels = self.data[sub_idx]
        zmax_x = zmax_arr[i]
        e4_x   = e4_arr[i]
        y      = int(labels[i])
        return torch.FloatTensor(zmax_x), torch.FloatTensor(e4_x), torch.tensor(y, dtype=torch.long)


print("\nBuilding datasets...")
train_ds = WearableDataset(TRAIN_SUBS, STUDENT_DATA_PATH)
print()
test_ds  = WearableDataset(TEST_SUBS, STUDENT_DATA_PATH)
print("Datasets ready.")


# ============================================================
# ENCODERS (same lightweight design as the KD student)
# ============================================================
class ZmaxEEGEncoder(nn.Module):
    """Compact multi-scale CNN + BiGRU for raw Zmax EEG (2ch, 1920 samples @ 64Hz)."""
    def __init__(self, in_ch=2, d_model=D_MODEL, dropout=DROPOUT):
        super().__init__()
        mid = d_model // 2

        def branch(kernel):
            return nn.Sequential(
                nn.Conv1d(in_ch, mid, kernel_size=kernel, stride=4, padding=kernel // 2),
                nn.BatchNorm1d(mid), nn.GELU(), nn.MaxPool1d(2, 2), nn.Dropout(dropout)
            )
        self.small, self.large = branch(15), branch(60)
        with torch.no_grad():
            dummy = torch.zeros(1, in_ch, 1920)
            L_s, L_l = self.small(dummy).shape[2], self.large(dummy).shape[2]
        target_L = min(L_s, L_l)
        self.pool_s = nn.AdaptiveAvgPool1d(target_L)
        self.pool_l = nn.AdaptiveAvgPool1d(target_L)
        self.proj = nn.Sequential(nn.Conv1d(2 * mid, d_model, kernel_size=1),
                                   nn.BatchNorm1d(d_model), nn.GELU())
        self.gru = nn.GRU(d_model, d_model // 2, batch_first=True, bidirectional=True)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        fs, fl = self.pool_s(self.small(x)), self.pool_l(self.large(x))
        feat = self.proj(torch.cat([fs, fl], dim=1)).permute(0, 2, 1)
        gru_out, _ = self.gru(feat)
        return self.norm(feat.mean(dim=1) + gru_out.mean(dim=1))


class E4Encoder(nn.Module):
    """1D-CNN + BiGRU for E4 (3ch: BVP, HR, TEMP, 1920 samples @ 64Hz)."""
    def __init__(self, in_ch=3, d_model=D_MODEL, dropout=DROPOUT):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(in_ch, d_model // 2, kernel_size=15, stride=4, padding=7),
            nn.BatchNorm1d(d_model // 2), nn.GELU(), nn.MaxPool1d(2, 2),
            nn.Conv1d(d_model // 2, d_model, kernel_size=8, padding=4),
            nn.BatchNorm1d(d_model), nn.GELU(), nn.MaxPool1d(2, 2),
            nn.Dropout(dropout),
        )
        self.gru = nn.GRU(d_model, d_model // 2, batch_first=True, bidirectional=True)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        feat = self.conv(x).permute(0, 2, 1)
        gru_out, _ = self.gru(feat)
        return self.norm(feat.mean(dim=1) + gru_out.mean(dim=1))


# ============================================================
# S1 MODEL: simple concat fusion (NO gate, NO artifact info)
# ============================================================
class WearableBaselineModel(nn.Module):
    def __init__(self, d_model=D_MODEL, n_classes=5, dropout=DROPOUT):
        super().__init__()
        self.zmax_enc = ZmaxEEGEncoder(in_ch=2, d_model=d_model, dropout=dropout)
        self.e4_enc   = E4Encoder(in_ch=3, d_model=d_model, dropout=dropout)
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model * 2), nn.Linear(d_model * 2, 64), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(64, n_classes)
        )

    def forward(self, zmax_x, e4_x):
        z = self.zmax_enc(zmax_x)
        e = self.e4_enc(e4_x)
        fused = torch.cat([z, e], dim=1)   # simple concatenation, no gating
        logits = self.classifier(fused)
        return logits


# ============================================================
# CLASS WEIGHTS
# ============================================================
cw_np = train_ds.label_counts.sum() / (5 * train_ds.label_counts)
cw = torch.FloatTensor(cw_np).to(device)
print(f"\nClass weights:")
for name, w in zip(LABEL_NAMES, cw_np):
    print(f"  {name}: {w:.3f}")


def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def train_epoch_fn(model, loader, optimizer, scheduler, criterion):
    model.train()
    total_loss = 0
    preds, labs = [], []
    for zmax_x, e4_x, y in loader:
        zmax_x, e4_x, y = zmax_x.to(device), e4_x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(zmax_x, e4_x)
        loss = criterion(logits, y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
        preds.extend(logits.argmax(1).cpu().numpy())
        labs.extend(y.cpu().numpy())
    n = len(loader)
    acc = accuracy_score(labs, preds)
    f1 = f1_score(labs, preds, average='macro', zero_division=0)
    return total_loss / n, acc, f1


def evaluate_fn(model, loader):
    model.eval()
    preds, labs = [], []
    with torch.no_grad():
        for zmax_x, e4_x, y in loader:
            logits = model(zmax_x.to(device), e4_x.to(device))
            preds.extend(logits.argmax(1).cpu().numpy())
            labs.extend(y.numpy())
    preds = np.array(preds); labs = np.array(labs)
    acc = accuracy_score(labs, preds)
    f1 = f1_score(labs, preds, average='macro', zero_division=0)
    kappa = cohen_kappa_score(labs, preds)
    per_cls = f1_score(labs, preds, average=None, zero_division=0)
    return acc, f1, kappa, per_cls


# ============================================================
# CSV SETUP
# ============================================================
csv_summary_path = os.path.join(EVAL_PATH, "summary.csv")
fields = ["seed", "acc", "f1_macro", "kappa",
          "f1_Wake", "f1_N1", "f1_N2", "f1_N3", "f1_REM"]
with open(csv_summary_path, 'w', newline='') as f:
    csv.DictWriter(f, fields).writeheader()

# ============================================================
# 5-SEED TRAINING LOOP
# ============================================================
all_results = []

for seed_idx, seed in enumerate(SEEDS):
    print(f"\n{'='*60}\nSEED {seed}  ({seed_idx+1}/{len(SEEDS)})\n{'='*60}")
    set_seed(seed)

    train_loader = DataLoader(
        train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0,
        generator=torch.Generator().manual_seed(seed)
    )
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    model = WearableBaselineModel().to(device)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"  Parameters : {n_params:,}")

    criterion = nn.CrossEntropyLoss(weight=cw)

    optimizer = optim.AdamW(model.parameters(), lr=5e-4, weight_decay=3e-4,
                             betas=(0.9, 0.98), eps=1e-9)
    total_steps = N_EPOCHS * len(train_loader)
    warmup_steps = int(0.05 * total_steps)

    def warmup_cosine(step):
        if step < warmup_steps:
            return step / max(warmup_steps, 1)
        t = (step - warmup_steps) / max(total_steps - warmup_steps, 1)
        return 0.5 * (1 + math.cos(math.pi * t))

    scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=warmup_cosine)

    best_f1 = 0.0
    best_path = os.path.join(EVAL_PATH, f"best_seed{seed}.pt")

    for epoch in range(1, N_EPOCHS + 1):
        tr_loss, tr_acc, tr_f1 = train_epoch_fn(
            model, train_loader, optimizer, scheduler, criterion
        )
        vl_acc, vl_f1, vl_kap, vl_per = evaluate_fn(model, test_loader)

        saved = ""
        if vl_f1 > best_f1:
            best_f1 = vl_f1
            torch.save(model.state_dict(), best_path)
            saved = " <- BEST"

        lr = optimizer.param_groups[0]['lr']
        print(f"  Ep[{epoch:02d}/{N_EPOCHS}] Loss:{tr_loss:.3f} "
              f"TrAcc:{tr_acc:.3f} ValAcc:{vl_acc:.3f} F1:{vl_f1:.3f} "
              f"k:{vl_kap:.3f} LR:{lr:.2e}{saved}")

    model.load_state_dict(torch.load(best_path, map_location=device))
    fin_acc, fin_f1, fin_kap, fin_per = evaluate_fn(model, test_loader)

    print(f"\n  Seed {seed} FINAL: Acc={fin_acc*100:.2f}% F1={fin_f1:.4f} k={fin_kap:.4f}")
    for i, name in enumerate(LABEL_NAMES):
        print(f"    {name:6s}: {fin_per[i]:.4f}")

    all_results.append({
        'seed': seed, 'acc': fin_acc, 'f1': fin_f1, 'kappa': fin_kap,
        'per_cls': fin_per
    })

    with open(csv_summary_path, 'a', newline='') as f:
        csv.DictWriter(f, fields).writerow({
            "seed": seed,
            "acc": round(fin_acc, 4), "f1_macro": round(fin_f1, 4),
            "kappa": round(fin_kap, 4),
            "f1_Wake": round(fin_per[0], 4), "f1_N1": round(fin_per[1], 4),
            "f1_N2": round(fin_per[2], 4), "f1_N3": round(fin_per[3], 4),
            "f1_REM": round(fin_per[4], 4),
        })

# ============================================================
# FINAL REPORT
# ============================================================
accs   = np.array([r['acc'] for r in all_results]) * 100
f1s    = np.array([r['f1'] for r in all_results])
kappas = np.array([r['kappa'] for r in all_results])

print(f"\n{'='*60}\nS1: WEARABLE-ONLY BASELINE -- 5-SEED REPORT\n{'='*60}")
print(f"Accuracy : {accs.mean():.2f} +- {accs.std():.2f}%")
print(f"Macro F1 : {f1s.mean():.4f} +- {f1s.std():.4f}")
print(f"Kappa    : {kappas.mean():.4f} +- {kappas.std():.4f}")

print("\nThis is your S1 reference point. Next step (S2) adds")
print("knowledge distillation from the frozen PSG teacher")
print("(MultiScaleSleepNetPlain ensemble, F1=0.8126) and we compare")
print("against this baseline to measure the benefit of KD alone.")

print(f"\nSummary saved: {csv_summary_path}")
print("Done!")

Device : cuda
S1: Wearable-only baseline (Zmax EEG + E4, plain CE, no KD, no gate)
Output : D:\22\AA\evaluation\teacher-student\s1_wearable_baseline
Split loaded -> Train:76  Test:20

Building datasets...
  Subjects: 71   Samples: 66,766

  Subjects: 19   Samples: 18,899
Datasets ready.

Class weights:
  Wake: 1.838
  N1: 3.149
  N2: 0.446
  N3: 1.009
  REM: 1.107

SEED 42  (1/5)
  Parameters : 153,989
  Ep[01/30] Loss:1.350 TrAcc:0.449 ValAcc:0.388 F1:0.365 k:0.206 LR:3.33e-04 <- BEST
  Ep[02/30] Loss:1.170 TrAcc:0.532 ValAcc:0.489 F1:0.429 k:0.295 LR:5.00e-04 <- BEST
  Ep[03/30] Loss:1.095 TrAcc:0.568 ValAcc:0.451 F1:0.424 k:0.293 LR:4.97e-04
  Ep[04/30] Loss:1.043 TrAcc:0.588 ValAcc:0.492 F1:0.442 k:0.320 LR:4.91e-04 <- BEST
  Ep[05/30] Loss:0.998 TrAcc:0.606 ValAcc:0.481 F1:0.438 k:0.322 LR:4.82e-04
  Ep[06/30] Loss:0.963 TrAcc:0.624 ValAcc:0.536 F1:0.471 k:0.371 LR:4.70e-04 <- BEST
  Ep[07/30] Loss:0.932 TrAcc:0.636 ValAcc:0.529 F1:0.480 k:0.370 LR:4.55e-04 <- BEST
  Ep[08/30] Los